Import necessary libraries

In [ ]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("heart.csv")

EDA

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.duplicated().sum()

In [ ]:
df["HeartDisease"].value_counts()

In [ ]:
num_col = ["Age","RestingBP","Cholesterol","FastingBS","MaxHR","Oldpeak","HeartDisease"]
for col in num_col:
    plt.figure(figsize=(6,4))
    sns.histplot(df[col],kde = True)


In [ ]:
df["Cholesterol"].value_counts()

In [ ]:
sns.heatmap(df.corr(numeric_only=True),annot = True)

In [ ]:
sns.countplot( x = df["Sex"] , hue = df['HeartDisease'])

In [ ]:
sns.countplot( x = df["ChestPainType"] , hue = df['HeartDisease'])

Data Preprocessing and Cleaning 


In [ ]:
ch_mean = df.loc[df["Cholesterol"] !=0 , "Cholesterol"].mean()
ch_mean

In [ ]:
df['Cholesterol'] = df["Cholesterol"].replace(0,ch_mean)
df["Cholesterol"] = df["Cholesterol"].round(2)

In [ ]:
df.head()

In [ ]:
df["Cholesterol"].value_counts()

In [ ]:
df["RestingBP"].value_counts().unique()

In [ ]:
resting_bp_mean = df.loc[df["RestingBP"] != 0 ,"RestingBP"].mean()

In [ ]:
df["RestingBP"] = df["RestingBP"].replace(0,resting_bp_mean)
df["RestingBP"] = df['RestingBP'].round(2)

In [ ]:
df.head()

In [ ]:
df_encoded = pd.get_dummies(df,drop_first = True )
df_encoded = df_encoded.astype(int)

In [ ]:
df_encoded.head()

Standard Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler
numeric_columns = ['Age' , 'RestingBP', 'Cholesterol','MaxHR','Oldpeak']
scaler = StandardScaler()
df_encoded[numeric_columns] = scaler.fit_transform(df_encoded[numeric_columns])

In [ ]:
df_encoded.head()

In [ ]:
x = df_encoded.drop("HeartDisease",axis = 1)
y = df_encoded["HeartDisease"]

Split the dataset

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.33, random_state= 42)

Training Multiple models and choosing that model which perform the best

In [ ]:
from sklearn.metrics import accuracy_score,f1_score,classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
x.head()

In [ ]:
models = {'Logistic Regression' : LogisticRegression(),
          "KNN" : KNeighborsClassifier(),
          "Naive Byes" : GaussianNB(),
          "Decision Tree" : DecisionTreeClassifier(),
          "SVM" : SVC(probability=True)}

In [ ]:
result = []

In [ ]:
for name , model in models.items():
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test,y_pred)
    result.append({
        'Model' : name,
        'Accuracy' : round(acc,4),
        'F1 Score' : round(f1,4)})


In [ ]:
result
#since Logistic Regression is performing best with 86% accuracy 
#so we use Logistic Regression model for prediction

now save this file as pikle

In [ ]:
import joblib 

In [ ]:
joblib.dump(models['Logistic Regression'],"Logistic_heart.pkl")
joblib.dump(scaler,"scaler.pkl")
joblib.dump(x.columns.tolist(),"columns.pkl")